## Research Artifact — Broken BM25 Baseline
This notebook intentionally preserves the original `.split()` BM25 tokenization.
The fixed camelCase-aware version lives in 02b_eval_baseline.ipynb.
The recall delta between the two notebooks provides evidence for the BM25 contribution.
Isolation is approximate because the QA pairs are unconfirmed and the diagnostic candidate pools differ.
Do not refactor or update this notebook.

# KernelPack RAG — BM25 Failure Mode (v1→v3)

Sidecar notebook. Demonstrates retrieval degradation from a broken BM25 leg: whitespace-only tokenization, no camelCase or snake_case splitting.

| Version | Change | recall@3 |
|---------|--------|----------|
| v1 | full index, raw source, MIN_LINES=5 | 1/10 (10%) |
| v2 | MIN_LINES=1 | 1/10 (10%) |
| v3a | solvers only, no summaries (control) | 2/10 (20%) |
| v3 | solvers only, LLM summaries | 4/10 (40%) |

**Fixed across all versions:** tree-sitter AST chunking, UniXcoder dense embeddings (ChromaDB, L2), BM25 whitespace tokenization, RRF (k=60), recall@3.  **Eval set:** `qa_pairs/solvers_qa.json` — 10 pairs, `kernelpack.solvers`. Tiers: api (4), workflow (3), conceptual (3).


## Setup


In [35]:
%pip install -q \
    sentence-transformers==5.5.0 \
    tree-sitter==0.25.2 \
    tree-sitter-python==0.25.0 \
    chromadb==1.5.9 \
    rank-bm25==0.2.2

Note: you may need to restart the kernel to use updated packages.


In [36]:
import json
import numpy as np
from pathlib import Path

import chromadb
import tree_sitter_python as tspython
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from tree_sitter import Language, Parser

# ── paths ──────────────────────────────────────────────────────────────────────
# Assumes kernelpack-python is cloned in the same parent directory as this repo.
# If not, update this path to point at your local kernelpack-python/src/kernelpack.
REPO_PATH = Path("../../kernelpack-python/src/kernelpack")
if not REPO_PATH.exists():
    raise FileNotFoundError(
        f"kernelpack-python not found at {REPO_PATH.resolve()}. "
        "Clone it as a sibling directory: git clone https://github.com/ShankarLab/kernelpack-python"
    )

# Index built against kernelpack-python @ b0cc141 (Add Euler convergence study)
# Do NOT git pull without rebuilding the index and re-running eval
KERNELPACK_COMMIT = "b0cc141"

# ── retrieval config ───────────────────────────────────────────────────────────
TOP_K = 3   # results returned per query

In [37]:
from utils.chunking import (
    load_docs, get_class_name, extract_chunks,
    make_chunk_id, make_metadata,
)
from utils.eval_utils import (
    is_hit, run_eval, print_recall, eval_query,
    compare_versions, diagnose_wide, diagnose_wide_hybrid,
)

In [38]:
# Load all .py files from the repo
docs = load_docs(REPO_PATH)

Loaded 34 files from ../../kernelpack-python/src/kernelpack


In [39]:
# CHUNKING
MIN_LINES = 5   # functions with fewer than 5 lines dropped (inclusive)

# tree-sitter parser setup
PY_LANGUAGE = Language(tspython.language())
parser_ts   = Parser(PY_LANGUAGE)


all_chunks, all_dropped = [], []
for doc in docs:
    tree = parser_ts.parse(doc["text"].encode("utf-8"))
    kept, dropped = extract_chunks(doc["text"], tree, doc["path"])
    all_chunks.extend(kept)
    all_dropped.extend(dropped)

print(f"Chunks kept   : {len(all_chunks)}")
print(f"Chunks dropped: {len(all_dropped)}  (functions < {MIN_LINES} lines)")

Chunks kept   : 337
Chunks dropped: 277  (functions < 5 lines)


## Embedding Model & ChromaDB Client


In [40]:
model = SentenceTransformer("microsoft/unixcoder-base")
model.max_seq_length = 512
ef = SentenceTransformerEmbeddingFunction(model_name="microsoft/unixcoder-base")
ef._model = model

client = chromadb.EphemeralClient()
print("Embedding model and ChromaDB client ready.")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 66222.35it/s]


Embedding model and ChromaDB client ready.


## Helper Functions


In [41]:
# ── Index builder ──────────────────────────────────────────────────────────────

def build_index(
    chunks: list[dict],
    collection_name: str,
    documents: list[str] | None = None,
) -> dict:
    """Build a ChromaDB collection + BM25 index from a list of chunks.

    Args:
        chunks:          Output of extract_chunks.
        collection_name: Name for the ChromaDB collection.
        documents:       Pre-built document strings (e.g. summary-enriched text).
                         If None, raw chunk source text is used.
    Returns:
        dict with keys: col (ChromaDB collection), bm25 (BM25Okapi), ids (list[str]).
    """
    if documents is None:
        documents = [c["text"] for c in chunks]

    raw_ids   = [make_chunk_id(c) for c in chunks]
    metadatas = [make_metadata(c) for c in chunks]

    # deduplicate by ID
    seen = set()
    deduped_docs, deduped_metas, deduped_ids = [], [], []
    for doc, meta, cid in zip(documents, metadatas, raw_ids):
        if cid not in seen:
            seen.add(cid)
            deduped_docs.append(doc)
            deduped_metas.append(meta)
            deduped_ids.append(cid)

    col = client.get_or_create_collection(collection_name, embedding_function=ef)
    col.add(documents=deduped_docs, metadatas=deduped_metas, ids=deduped_ids)
    print(f"Collection {collection_name!r}: {col.count()} chunks")

    bm25 = BM25Okapi([doc.split() for doc in deduped_docs])
    return {"col": col, "bm25": bm25, "ids": deduped_ids}


# ── Retrieval ──────────────────────────────────────────────────────────────────

def retrieve(query: str, index: dict, k: int = TOP_K, silent: bool = False) -> list[dict]:
    """Hybrid BM25 + dense retrieval with RRF fusion.

    Args:
        query:  Query string.
        index:  Dict with keys col, bm25, ids — output of build_index.
        k:      Number of results to return.
        silent: If True, suppresses printed output.
    Returns:
        Ordered list of metadata dicts for the top-k chunks.
    """
    col  = index["col"]
    bm25 = index["bm25"]
    ids  = index["ids"]

    # dense leg — embed with UniXcoder, return top 10 by L2 distance
    dense_res = col.query(query_texts=[query], n_results=10)
    dense_ids = dense_res["ids"][0]
    dense_l2  = {cid: d for cid, d in zip(dense_res["ids"][0], dense_res["distances"][0])}

    # BM25 leg — scored by keyword overlap, return top 10
    bm25_scores = bm25.get_scores(query.split())
    top_bm25    = sorted(range(len(bm25_scores)), key=lambda i: bm25_scores[i], reverse=True)[:10]
    bm25_ids    = [ids[i] for i in top_bm25]

    # RRF fusion — score += 1/(rank + 60) for each list the chunk appears in
    # k=60 is the smoothing constant; dampens the advantage of being ranked 1st
    rrf: dict[str, float] = {}
    for rank, cid in enumerate(dense_ids):
        rrf[cid] = rrf.get(cid, 0.0) + 1 / (rank + 60)
    for rank, cid in enumerate(bm25_ids):
        rrf[cid] = rrf.get(cid, 0.0) + 1 / (rank + 60)

    top_ids = sorted(rrf, key=rrf.__getitem__, reverse=True)[:k]

    # col.get() returns results in input-ID order — reorder to match RRF ranking
    fetched   = col.get(ids=top_ids, include=["documents", "metadatas"])
    id_to_idx = {cid: i for i, cid in enumerate(fetched["ids"])}
    ordered_ids   = [cid for cid in top_ids if cid in id_to_idx]
    ordered_docs  = [fetched["documents"][id_to_idx[cid]] for cid in ordered_ids]
    ordered_metas = [fetched["metadatas"][id_to_idx[cid]] for cid in ordered_ids]

    if not silent:
        print(f'\nQUERY: "{query}"')
        print("=" * 60)
        for i, (cid, doc, meta) in enumerate(zip(ordered_ids, ordered_docs, ordered_metas), 1):
            cls    = f"{meta['class_name']}." if meta["class_name"] else ""
            l2_str = f"{dense_l2[cid]:.4f}" if cid in dense_l2 else "n/a (BM25-only)"
            print(f"\n  Rank {i} | RRF {rrf[cid]:.4f} | L2 {l2_str}")
            print(f"  {meta['file_path']} — {cls}{meta['function_name']} (lines {meta['start_line']}–{meta['end_line']})")
            print("  " + "-" * 56)
            print("  " + doc[:300].replace("\n", "\n  "))

    return ordered_metas

def retrieve_wide(query: str, index: dict, n: int = 50) -> list[dict]:
    """Hybrid BM25 + dense with expanded candidate pool. Used by diagnose_wide_hybrid."""
    col  = index["col"]
    bm25 = index["bm25"]
    ids  = index["ids"]

    dense_res = col.query(query_texts=[query], n_results=n)
    dense_ids = dense_res["ids"][0]

    bm25_scores = bm25.get_scores(query.split())  # broken tokenizer — intentional
    top_bm25    = sorted(range(len(bm25_scores)), key=lambda i: bm25_scores[i], reverse=True)[:n]
    bm25_ids    = [ids[i] for i in top_bm25]

    rrf: dict[str, float] = {}
    for rank, cid in enumerate(dense_ids):
        rrf[cid] = rrf.get(cid, 0.0) + 1 / (rank + 60)
    for rank, cid in enumerate(bm25_ids):
        rrf[cid] = rrf.get(cid, 0.0) + 1 / (rank + 60)

    top_ids = sorted(rrf, key=rrf.__getitem__, reverse=True)[:n]
    fetched   = col.get(ids=top_ids, include=["metadatas"])
    id_to_idx = {cid: i for i, cid in enumerate(fetched["ids"])}
    return [fetched["metadatas"][id_to_idx[cid]] for cid in top_ids if cid in id_to_idx]

print("Helper functions defined.")

Helper functions defined.


## Load Eval Data


In [42]:
with open("qa_pairs/solvers_qa.json") as f:
    qa_pairs_solvers = json.load(f)

with open("qa_pairs/benchmark_qa.json") as f:
    qa_pairs_benchmark = json.load(f)

print(f"Solvers QA pairs  : {len(qa_pairs_solvers)}")
print(f"Benchmark QA pairs: {len(qa_pairs_benchmark)}")

Solvers QA pairs  : 10
Benchmark QA pairs: 3


## v1 — Baseline

MIN_LINES=5, 337 chunks, raw source text only.


In [43]:
idx_v1 = build_index(all_chunks, "kp-hybrid-v1")

Collection 'kp-hybrid-v1': 337 chunks


In [44]:
for qa in qa_pairs_solvers:
    eval_query(qa, idx_v1)


QUERY : For a publication benchmark with mixed Dirichlet and Neumann elliptic boundary data, how can I obtain the physical field together with the assembled matrix, right-hand side pieces, and nullspace metadata needed for reproducibility?
TIER  : api
TARGET: PoissonSolver.solve (src/kernelpack/solvers/poisson.py)
ANSWER: The stationary Poisson API materializes the Neumann and Dirichlet boundary coefficients, assembles the boundary operator for those coefficients, evaluates the forcing on interior-plus-boundary nodes, and evaluates the boundary right-hand side. It treats the case as pure Neumann when the maximum absolute Dirichlet coefficient is at most 1e-13, augments the sparse system with a constant-mode constraint in that case, and uses a direct sparse solve unless an initial guess triggers GMRES with fallback. The result dictionary contains `u`, `full_state`, `L`, `BC`, `system_matrix`, `rhs`, `target_rhs`, `boundary_rhs`, `used_nullspace_augmentation`, and `lagrange_multiplier`.

In [45]:
for qa in qa_pairs_benchmark:
    eval_query(qa, idx_v1)


QUERY : how to create an RBF-FD differentiation matrix?
TIER  : workflow
TARGET: FDDiffOp.assemble_op (src/kernelpack/rbffd/core.py)
ANSWER: Use the RBF-FD classes in `kernelpack.rbffd`. Create stencil settings with `StencilProperties.from_accuracy(operator=..., convergence_order=..., dimension=..., approximation="rbf", tree_mode="all", point_set="interior_boundary")`, create operator settings with `OpProperties(...)`, then instantiate `FDDiffOp(lambda: RBFStencil())`. Call `assemble_op(domain, op_name, st_props, op_props, neu_coeff=None, dir_coeff=None, active_rows=None)`, where `op_name` is a string such as `"lap"`, `"grad"`, or `"bc"` as supported by the stencil code. Retrieve the sparse CSR differentiation matrix with `get_op()`. For overlapped assembly, the source also provides `FDODiffOp` with the same `assemble_op` and `get_op` workflow.
──────────────────────────────────────────────────────────────────────
  Rank 1: divfree/core.py — DivFreeGram
  Rank 2: divfree/core.py — DFP

In [46]:
hits_v1, totals_v1 = run_eval(qa_pairs_solvers, idx_v1)
print_recall(hits_v1, totals_v1, k=TOP_K, label="v1 — solvers")

── Recall@3 by tier — v1 — solvers
  api          0/4  (0%)
  workflow     0/3  (0%)
  conceptual   1/3  (33%)
  overall      1/10  (10%)


In [47]:
hits_bench_v1, totals_bench_v1 = run_eval(qa_pairs_benchmark, idx_v1)
print_recall(hits_bench_v1, totals_bench_v1, k=TOP_K, label="v1 — benchmark")

── Recall@3 by tier — v1 — benchmark
  api          0/0  (0%)
  workflow     0/3  (0%)
  conceptual   0/0  (0%)
  overall      0/3  (0%)


In [48]:
# Wide retrieval diagnostic (top-50): dense-only then hybrid.
# dense rank: embedding quality in isolation.
# hybrid rank: BM25 + RRF on top — delta vs dense isolates BM25 contribution.
# mentioned-in-top-10: target symbol appears in a retrieved chunk body (context reachability).
print("── dense ──")
diagnose_wide(qa_pairs_solvers, idx_v1)
print("\n── hybrid ──")
diagnose_wide_hybrid(qa_pairs_solvers, idx_v1)

── dense ──
  PoissonSolver.solve                                           dense: 18          mentioned-in-top-10: ✓
  VariablePoissonSolver.solve                                   dense: 41          mentioned-in-top-10: ✓
  NonlinearVariablePoissonSolver.solve                          dense: 47          mentioned-in-top-10: ✓
  PUSLAdvectionSolver.forward_sl_step                           dense: 3           mentioned-in-top-10: ✓
  DiffusionSolver.set_state_history                             dense: 1           mentioned-in-top-10: ✓
  MultiSpeciesDiffusionSolver.set_initial_state                 dense: 4           mentioned-in-top-10: ✓
  PUSLFDAdvectionDiffusionSolver.bdf3_step                      dense: 2           mentioned-in-top-10: ✓
  build_stencil_properties                                      dense: 2           mentioned-in-top-10: ✓
  pu_patch_weight                                               dense: 29          mentioned-in-top-10: ✓
  IncompressibleEulerBDFBackend._g

## v1 — Failure Mode Analysis

Wide retrieval (top-50), dense-only and hybrid, to classify each miss.

Note: Hybrid ranks in this table use expanded wide retrieval (top-50 dense + top-50 BM25) and are not directly comparable to recall@3, which uses top-10 dense + top-10 BM25.

> Output scan comment: no discrepancies found among the existing v1 dense-rank table values. The v1 hybrid column below was added from the executed `diagnose_wide_hybrid` output.

| Target | v1 dense | v1 hybrid | Category |
|---|---|---|---|
| PoissonSolver.solve | 18 | 3 | Reranker |
| VariablePoissonSolver.solve | 41 | not found | BM25 failure |
| NonlinearVariablePoissonSolver.solve | 47 | not found | BM25 failure |
| PUSLAdvectionSolver.forward_sl_step | 3 | 14 | BM25 failure |
| DiffusionSolver.set_state_history | 1 | 4 | BM25 drag |
| MultiSpeciesDiffusionSolver.set_initial_state | 4 | 11 | BM25 failure |
| PUSLFDAdvectionDiffusionSolver.bdf3_step | 2 | 10 | BM25 failure |
| build_stencil_properties | 2 | 1 | Hit |
| pu_patch_weight | 29 | not found | BM25 failure |
| IncompressibleEulerBDFBackend._get_cached_system | 15 | 31 | BM25 failure |

BM25 failures are visible where dense finds the target but the whitespace-tokenized hybrid path pushes it down or loses it. This is the baseline failure mode for this notebook.

## v1 — Result

### recall@3: **1/10 (10%)**
| Tier | Hits |
|------|------|
| api | 0/4 |
| workflow | 0/3 |
| conceptual | 1/3 |

### qa_solvers.json - k@3, 5, 10
| k | recall (hybrid) |
|---|--------|
| 3 | 1/10 |
| 5 | 4/10 |
| 10 | 5/10 |

**BM25 failure signal:** 4 targets sit at dense-only ranks 1–4
(`set_state_history`, `bdf3_step`, `forward_sl_step`, `set_initial_state`)
but are absent from the hybrid top-3. BM25 scores them zero — it can't
match whitespace-tokenized query words against snake_case identifiers —
so other chunks with BM25 hits leapfrog them in RRF fusion.

## v2 — MIN_LINES=1

Drops the minimum line filter from 5 to 1. Tests whether short target functions were missing; adds noise from 277 additional short chunks (337 → 614).


In [49]:
all_chunks_v2, all_dropped_v2 = [], []
for doc in docs:
    tree = parser_ts.parse(doc["text"].encode("utf-8"))
    kept, dropped = extract_chunks(doc["text"], tree, doc["path"], min_lines=1)
    all_chunks_v2.extend(kept)
    all_dropped_v2.extend(dropped)

print(f"v1 chunks: {len(all_chunks)}")
print(f"v2 chunks: {len(all_chunks_v2)}")

idx_v2 = build_index(all_chunks_v2, "kp-hybrid-v2")

v1 chunks: 337
v2 chunks: 614
Collection 'kp-hybrid-v2': 614 chunks


In [50]:
# what does the added noise look like?
new_chunks = [c for c in all_chunks_v2 if (c["end_line"] - c["start_line"] + 1) < 5]
for c in new_chunks[:20]:
    lines = c["end_line"] - c["start_line"] + 1
    print(f"{lines} lines | {c['class_name']}.{c['function_name']}")
    print("  " + c["text"][:120].replace("\n", "\n  "))
    print()

3 lines | None._stack_field
  def _stack_field(U: np.ndarray) -> np.ndarray:
      U = np.asarray(U, dtype=float)
      return np.concatenate([U[:, d] for

4 lines | None._unstack_field
  def _unstack_field(rhs: np.ndarray, dim: int) -> np.ndarray:
      rhs = np.asarray(rhs, dtype=float).reshape(-1)
      n = 

4 lines | LocalDivFreeInterpolator._query_stencil_indices
  def _query_stencil_indices(self, point: np.ndarray) -> np.ndarray:
          idx = self.Tree.query(point, k=self.StencilSi

4 lines | LocalDivFreeInterpolator.assign_centers
  def assign_centers(self, Xq: np.ndarray) -> np.ndarray:
          Xq = np.asarray(Xq, dtype=float)
          idx = self.Tree

2 lines | DomainDescriptor.set_outer_level_set
  def set_outer_level_set(self, level_set: object) -> None:
          self.outer_level_set = level_set

2 lines | DomainDescriptor.set_sep_rad
  def set_sep_rad(self, sep_rad: float) -> None:
          self.sep_rad = float(sep_rad)

2 lines | DomainDescriptor.set_boundary_leve

In [51]:
for qa in qa_pairs_solvers:
    eval_query(qa, idx_v2)


QUERY : For a publication benchmark with mixed Dirichlet and Neumann elliptic boundary data, how can I obtain the physical field together with the assembled matrix, right-hand side pieces, and nullspace metadata needed for reproducibility?
TIER  : api
TARGET: PoissonSolver.solve (src/kernelpack/solvers/poisson.py)
ANSWER: The stationary Poisson API materializes the Neumann and Dirichlet boundary coefficients, assembles the boundary operator for those coefficients, evaluates the forcing on interior-plus-boundary nodes, and evaluates the boundary right-hand side. It treats the case as pure Neumann when the maximum absolute Dirichlet coefficient is at most 1e-13, augments the sparse system with a constant-mode constraint in that case, and uses a direct sparse solve unless an initial guess triggers GMRES with fallback. The result dictionary contains `u`, `full_state`, `L`, `BC`, `system_matrix`, `rhs`, `target_rhs`, `boundary_rhs`, `used_nullspace_augmentation`, and `lagrange_multiplier`.

In [52]:
for qa in qa_pairs_benchmark:
    eval_query(qa, idx_v2)


QUERY : how to create an RBF-FD differentiation matrix?
TIER  : workflow
TARGET: FDDiffOp.assemble_op (src/kernelpack/rbffd/core.py)
ANSWER: Use the RBF-FD classes in `kernelpack.rbffd`. Create stencil settings with `StencilProperties.from_accuracy(operator=..., convergence_order=..., dimension=..., approximation="rbf", tree_mode="all", point_set="interior_boundary")`, create operator settings with `OpProperties(...)`, then instantiate `FDDiffOp(lambda: RBFStencil())`. Call `assemble_op(domain, op_name, st_props, op_props, neu_coeff=None, dir_coeff=None, active_rows=None)`, where `op_name` is a string such as `"lap"`, `"grad"`, or `"bc"` as supported by the stencil code. Retrieve the sparse CSR differentiation matrix with `get_op()`. For overlapped assembly, the source also provides `FDODiffOp` with the same `assemble_op` and `get_op` workflow.
──────────────────────────────────────────────────────────────────────
  Rank 1: divfree/core.py — DivFreeGram
  Rank 2: solvers/pu_sl_pu_adve

In [53]:
hits_v2, totals_v2 = run_eval(qa_pairs_solvers, idx_v2)
print_recall(hits_v2, totals_v2, k=TOP_K, label="v2 — solvers")

print("── v1 vs v2 recall@3 ─────────────────")
print(f"  v1 overall: {sum(hits_v1.values())}/10 ({round(100 * sum(hits_v1.values()) / 10)}%)")
print(f"  v2 overall: {sum(hits_v2.values())}/10 ({round(100 * sum(hits_v2.values()) / 10)}%)")

── Recall@3 by tier — v2 — solvers
  api          0/4  (0%)
  workflow     0/3  (0%)
  conceptual   1/3  (33%)
  overall      1/10  (10%)
── v1 vs v2 recall@3 ─────────────────
  v1 overall: 1/10 (10%)
  v2 overall: 1/10 (10%)


In [54]:
hits_bench_v2, totals_bench_v2 = run_eval(qa_pairs_benchmark, idx_v2)
print_recall(hits_bench_v2, totals_bench_v2, k=TOP_K, label="v2 — benchmark")

── Recall@3 by tier — v2 — benchmark
  api          0/0  (0%)
  workflow     0/3  (0%)
  conceptual   0/0  (0%)
  overall      0/3  (0%)


In [55]:
# log any regressions or recoveries between versions (hybrid retrieval)
compare_versions(qa_pairs_solvers, idx_v1, idx_v2, "v1", "v2")

In [56]:
# Wide retrieval diagnostic (top-50): dense-only then hybrid.
# dense rank: embedding quality in isolation.
# hybrid rank: BM25 + RRF on top — delta vs dense isolates BM25 contribution.
# mentioned-in-top-10: target symbol appears in a retrieved chunk body (context reachability).
print("── dense ──")
diagnose_wide(qa_pairs_solvers, idx_v2)
print("\n── hybrid ──")
diagnose_wide_hybrid(qa_pairs_solvers, idx_v2)

── dense ──
  PoissonSolver.solve                                           dense: 33          mentioned-in-top-10: ✓
  VariablePoissonSolver.solve                                   dense: not found   mentioned-in-top-10: ✗
  NonlinearVariablePoissonSolver.solve                          dense: not found   mentioned-in-top-10: ✓
  PUSLAdvectionSolver.forward_sl_step                           dense: 5           mentioned-in-top-10: ✓
  DiffusionSolver.set_state_history                             dense: 7           mentioned-in-top-10: ✓
  MultiSpeciesDiffusionSolver.set_initial_state                 dense: 7           mentioned-in-top-10: ✓
  PUSLFDAdvectionDiffusionSolver.bdf3_step                      dense: 7           mentioned-in-top-10: ✓
  build_stencil_properties                                      dense: 2           mentioned-in-top-10: ✓
  pu_patch_weight                                               dense: 42          mentioned-in-top-10: ✓
  IncompressibleEulerBDFBackend._g

## v2 — Failure Mode Analysis

Wide retrieval (top-50), compared against v1.

Note: Hybrid ranks in this table use expanded wide retrieval (top-50 dense + top-50 BM25) and are not directly comparable to recall@3, which uses top-10 dense + top-10 BM25.

> Output scan comment: no discrepancies found among the existing v1/v2 dense-rank table values. The v1/v2 hybrid columns below were added from the executed `diagnose_wide_hybrid` outputs.

| Target | v1 dense | v1 hybrid | v2 dense | v2 hybrid | Category |
|---|---|---|---|---|---|
| PoissonSolver.solve | 18 | 3 | 33 | 4 | BM25 drag |
| VariablePoissonSolver.solve | 41 | not found | not found | not found | **Regression** |
| NonlinearVariablePoissonSolver.solve | 47 | not found | not found | not found | Regression |
| PUSLAdvectionSolver.forward_sl_step | 3 | 14 | 5 | 14 | BM25 failure |
| DiffusionSolver.set_state_history | 1 | 4 | 7 | 4 | BM25 drag |
| MultiSpeciesDiffusionSolver.set_initial_state | 4 | 11 | 7 | 9 | BM25 failure |
| PUSLFDAdvectionDiffusionSolver.bdf3_step | 2 | 10 | 7 | 14 | BM25 failure |
| build_stencil_properties | 2 | 1 | 2 | 1 | Hit |
| pu_patch_weight | 29 | not found | 42 | not found | BM25 failure |
| IncompressibleEulerBDFBackend._get_cached_system | 15 | 31 | 19 | 36 | BM25 failure |

BM25 failures persist. Noise from short chunks degraded the dense rankings for the BM25-sensitive targets, and `VariablePoissonSolver.solve` is explicitly `not found` in both the v2 dense and v2 hybrid outputs.

## v2 — Result

### recall@3: **1/10 (10%)**
| Tier | Hits |
|------|------|
| api | 0/4 |
| workflow | 0/3 |
| conceptual | 1/3 |

### qa_solvers.json - k@3, 5, 10
| k | recall (hybrid)|
|---|--------|
| 3 | 1/10 |
| 5 | 1/10 |
| 10 | 2/10 |

Short-function noise degraded dense rankings. recall@5 dropped from 4/10 to 1/10.

## v3a — Solvers-only (Control)

Index scoped to `kernelpack.solvers` (187 chunks), no summaries. Isolates the scoping effect from summary contribution.


In [57]:
# solvers-only chunk list — shared by v3a and v3
all_chunks_v3 = [c for c in all_chunks if "solvers" in c["path"]]
print(f"Solvers chunks: {len(all_chunks_v3)}")

Solvers chunks: 187


In [58]:
# v3a — solvers-only index, no summaries
# control experiment: isolates whether scoping to solvers alone improves recall
# independent of summary enrichment
idx_v3a = build_index(all_chunks_v3, "kp-hybrid-v3a")

hits_v3a, totals_v3a = run_eval(qa_pairs_solvers, idx_v3a)
print_recall(hits_v3a, totals_v3a, k=TOP_K, label="v3a — solvers")

print("── v1 vs v2 vs v3a recall@3 ────────────────")
print(f"  v1 overall: {sum(hits_v1.values())}/10 ({round(100 * sum(hits_v1.values()) / 10)}%)")
print(f"  v2 overall: {sum(hits_v2.values())}/10 ({round(100 * sum(hits_v2.values()) / 10)}%)")
print(f"  v3a (solvers only, no summaries) : {sum(hits_v3a.values())}/10 ({round(100 * sum(hits_v3a.values()) / 10)}%)")

Collection 'kp-hybrid-v3a': 187 chunks
── Recall@3 by tier — v3a — solvers
  api          0/4  (0%)
  workflow     1/3  (33%)
  conceptual   1/3  (33%)
  overall      2/10  (20%)
── v1 vs v2 vs v3a recall@3 ────────────────
  v1 overall: 1/10 (10%)
  v2 overall: 1/10 (10%)
  v3a (solvers only, no summaries) : 2/10 (20%)


In [59]:
compare_versions(qa_pairs_solvers, idx_v2, idx_v3a, "v2", "v3a")

✗ -> ✓ RECOVERED: DiffusionSolver.set_state_history


In [60]:
# Wide retrieval diagnostic (top-50): dense-only then hybrid.
# dense rank: embedding quality in isolation.
# hybrid rank: BM25 + RRF on top — delta vs dense isolates BM25 contribution.
# mentioned-in-top-10: target symbol appears in a retrieved chunk body (context reachability).
print("── dense ──")
diagnose_wide(qa_pairs_solvers, idx_v3a)
print("\n── hybrid ──")
diagnose_wide_hybrid(qa_pairs_solvers, idx_v3a)

── dense ──
  PoissonSolver.solve                                           dense: 15          mentioned-in-top-10: ✓
  VariablePoissonSolver.solve                                   dense: 30          mentioned-in-top-10: ✓
  NonlinearVariablePoissonSolver.solve                          dense: 40          mentioned-in-top-10: ✓
  PUSLAdvectionSolver.forward_sl_step                           dense: 2           mentioned-in-top-10: ✓
  DiffusionSolver.set_state_history                             dense: 1           mentioned-in-top-10: ✓
  MultiSpeciesDiffusionSolver.set_initial_state                 dense: 4           mentioned-in-top-10: ✓
  PUSLFDAdvectionDiffusionSolver.bdf3_step                      dense: 2           mentioned-in-top-10: ✓
  build_stencil_properties                                      dense: 1           mentioned-in-top-10: ✓
  pu_patch_weight                                               dense: 16          mentioned-in-top-10: ✓
  IncompressibleEulerBDFBackend._g

## v3a — Result

recall@3: **2/10 (20%)**

| k | recall |
|---|--------|
| 3 | 2/10 |
| 5 | 4/10 |
| 10 | 5/10 |


## v3a — Failure Mode Analysis

Wide retrieval (top-50), compared against v1 and v2 baselines.

Note: Hybrid ranks in this table use expanded wide retrieval (top-50 dense + top-50 BM25) and are not directly comparable to recall@3, which uses top-10 dense + top-10 BM25.

> Output scan comment: corrected the inherited v1 dense value for `VariablePoissonSolver.solve` from 40 to 41 to match the executed v1 `diagnose_wide` output. All v3a dense and hybrid ranks below match the executed v3a diagnostic outputs.

| Target | v1 dense | v1 hybrid | v2 dense | v2 hybrid | v3a dense | v3a hybrid | Category |
|---|---|---|---|---|---|---|---|
| PoissonSolver.solve | 18 | 3 | 33 | 4 | 15 | 4 | BM25 drag |
| VariablePoissonSolver.solve | 41 | not found | not found | not found | 30 | 20 | Reranker problem |
| NonlinearVariablePoissonSolver.solve | 47 | not found | not found | not found | 40 | not found | BM25 failure |
| PUSLAdvectionSolver.forward_sl_step | 3 | 14 | 5 | 14 | 2 | 7 | BM25 drag |
| DiffusionSolver.set_state_history | 1 | 4 | 7 | 4 | 1 | 3 | Hit |
| MultiSpeciesDiffusionSolver.set_initial_state | 4 | 11 | 7 | 9 | 4 | 8 | BM25 failure |
| PUSLFDAdvectionDiffusionSolver.bdf3_step | 2 | 10 | 7 | 14 | 2 | 17 | BM25 failure |
| build_stencil_properties | 2 | 1 | 2 | 1 | 1 | 1 | Hit |
| pu_patch_weight | 29 | not found | 42 | not found | 16 | 36 | BM25 drag |
| IncompressibleEulerBDFBackend._get_cached_system | 15 | 31 | 19 | 36 | 14 | 32 | BM25 failure |

Scoping to solvers reduced dense-rank noise for several targets, but the whitespace BM25 failure mode remains visible in the hybrid ranks.

## v3 — LLM Summaries

Prepends Codex-generated plain-English summaries to each solver chunk. 187 chunks (solvers-only), 187/187 matched a summary.


In [61]:
with open("metadata/solvers_summaries.json") as f:
    summaries = json.load(f)

# key = (function_name, class_name)
summary_lookup = {
    (s["function_name"], s["class_name"] or ""): s["summary"]
    for s in summaries
}
print(f"Loaded {len(summary_lookup)} summaries")

Loaded 366 summaries


In [62]:
# build enriched documents for v3 — prepend LLM summary to each chunk
documents_v3 = []
for c in all_chunks_v3:
    key     = (c["function_name"], c["class_name"] or "")
    summary = summary_lookup.get(key, "")
    enriched = f"{summary}\n\n{c['text']}" if summary else c["text"]
    documents_v3.append(enriched)

matched_v3 = sum(1 for c in all_chunks_v3
                 if (c["function_name"], c["class_name"] or "") in summary_lookup)
print(f"{matched_v3}/{len(all_chunks_v3)} chunks matched a summary")

187/187 chunks matched a summary


In [63]:
idx_v3 = build_index(all_chunks_v3, "kp-hybrid-v3", documents=documents_v3)

# sanity check: confirm summary is prepended to raw source
sample = idx_v3["col"].get(limit=1, include=["documents"])
print("\n── Sample enriched document (first 500 chars) ──")
print(sample["documents"][0][:500])

Collection 'kp-hybrid-v3': 187 chunks

── Sample enriched document (first 500 chars) ──
Translates a solver accuracy/order request and domain dimension into the stencil size, polynomial degree, spline degree, tree mode, and point set used by the RBF-FD assembler.

def build_stencil_properties(domain: DomainDescriptor, xi: int, theta: int, point_set: str) -> StencilProperties:
    # The solver layer asks for target order `xi` and operator order `theta`;
    # here we translate that request into a concrete stencil size and
    # polynomial degree that the rbffd layer understands.
   


In [64]:
hits_v3, totals_v3 = run_eval(qa_pairs_solvers, idx_v3)
print_recall(hits_v3, totals_v3, k=TOP_K, label="v3 — solvers")

print("── v1 vs v2 vs v3a vs v3 recall@3 ────────────────")
print(f"  v1 overall (full index, no summaries): {sum(hits_v1.values())}/10 ({round(100 * sum(hits_v1.values()) / 10)}%)")
print(f"  v2 overall (full index, min_lines = 1): {sum(hits_v2.values())}/10 ({round(100 * sum(hits_v2.values()) / 10)}%)")
print(f"  v3a (solvers only, no summaries) : {sum(hits_v3a.values())}/10 ({round(100 * sum(hits_v3a.values()) / 10)}%)")
print(f"  v3  (solvers only, with summaries) overall: {sum(hits_v3.values())}/10 ({round(100 * sum(hits_v3.values()) / 10)}%)")

── Recall@3 by tier — v3 — solvers
  api          1/4  (25%)
  workflow     1/3  (33%)
  conceptual   2/3  (67%)
  overall      4/10  (40%)
── v1 vs v2 vs v3a vs v3 recall@3 ────────────────
  v1 overall (full index, no summaries): 1/10 (10%)
  v2 overall (full index, min_lines = 1): 1/10 (10%)
  v3a (solvers only, no summaries) : 2/10 (20%)
  v3  (solvers only, with summaries) overall: 4/10 (40%)


In [65]:
compare_versions(qa_pairs_solvers, idx_v1, idx_v3, "v1", "v3")

✗ -> ✓ RECOVERED: PoissonSolver.solve
✗ -> ✓ RECOVERED: DiffusionSolver.set_state_history
✗ -> ✓ RECOVERED: pu_patch_weight


In [66]:
# Wide retrieval diagnostic (top-50): dense-only then hybrid.
# dense rank: embedding quality in isolation.
# hybrid rank: BM25 + RRF on top — delta vs dense isolates BM25 contribution.
# mentioned-in-top-10: target symbol appears in a retrieved chunk body (context reachability).
print("── dense ──")
diagnose_wide(qa_pairs_solvers, idx_v3)
print("\n── hybrid ──")
diagnose_wide_hybrid(qa_pairs_solvers, idx_v3)

── dense ──
  PoissonSolver.solve                                           dense: 6           mentioned-in-top-10: ✓
  VariablePoissonSolver.solve                                   dense: 18          mentioned-in-top-10: ✓
  NonlinearVariablePoissonSolver.solve                          dense: 18          mentioned-in-top-10: ✓
  PUSLAdvectionSolver.forward_sl_step                           dense: 4           mentioned-in-top-10: ✓
  DiffusionSolver.set_state_history                             dense: 4           mentioned-in-top-10: ✓
  MultiSpeciesDiffusionSolver.set_initial_state                 dense: 2           mentioned-in-top-10: ✓
  PUSLFDAdvectionDiffusionSolver.bdf3_step                      dense: 6           mentioned-in-top-10: ✓
  build_stencil_properties                                      dense: 1           mentioned-in-top-10: ✓
  pu_patch_weight                                               dense: 1           mentioned-in-top-10: ✓
  IncompressibleEulerBDFBackend._g

## v3 — Results

### Recall@3
| Version | Change | recall@3 |
|---------|--------|----------|
| v1 | full index, no summaries | 1/10 (10%) |
| v2 | MIN_LINES=1 | 1/10 (10%) |
| v3a | solvers only, no summaries | 2/10 (20%) |
| v3 | solvers only, with summaries | 4/10 (40%) |

| k | recall |
|---|--------|
| 3 | 4/10 |
| 5 | 5/10 |
| 10 | 6/10 |

| Tier | Hits |
|------|------|
| api | 1/4 |
| workflow | 1/3 |
| conceptual | 2/3 |


## v3 — Failure Mode Analysis

Wide retrieval (top-50), compared against v1, v2, and v3a baselines.

Note: Hybrid ranks in this table use expanded wide retrieval (top-50 dense + top-50 BM25) and are not directly comparable to recall@3, which uses top-10 dense + top-10 BM25.

> Output scan comment: no discrepancies found among the existing v1/v2/v3a/v3 dense-rank table values. The hybrid columns below were added from the executed `diagnose_wide_hybrid` outputs.

| Target | v1 dense | v1 hybrid | v2 dense | v2 hybrid | v3a dense | v3a hybrid | v3 dense | v3 hybrid | Category |
|---|---|---|---|---|---|---|---|---|---|
| PoissonSolver.solve | 18 | 3 | 33 | 4 | 15 | 4 | 6 | 2 | Hit |
| VariablePoissonSolver.solve | 41 | not found | not found | not found | 30 | 20 | 18 | 8 | Reranker problem |
| NonlinearVariablePoissonSolver.solve | 47 | not found | not found | not found | 40 | not found | 18 | 4 | BM25 drag |
| PUSLAdvectionSolver.forward_sl_step | 3 | 14 | 5 | 14 | 2 | 7 | 4 | 4 | BM25 drag |
| DiffusionSolver.set_state_history | 1 | 4 | 7 | 4 | 1 | 3 | 4 | 2 | Hit |
| MultiSpeciesDiffusionSolver.set_initial_state | 4 | 11 | 7 | 9 | 4 | 8 | 2 | 24 | BM25 failure |
| PUSLFDAdvectionDiffusionSolver.bdf3_step | 2 | 10 | 7 | 14 | 2 | 17 | 6 | 12 | BM25 failure |
| build_stencil_properties | 2 | 1 | 2 | 1 | 1 | 1 | 1 | 1 | Hit |
| pu_patch_weight | 29 | not found | 42 | not found | 16 | 36 | 1 | 1 | Hit |
| IncompressibleEulerBDFBackend._get_cached_system | 15 | 31 | 19 | 36 | 14 | 32 | 7 | 9 | Reranker problem |

Summaries add BM25-matchable plain text and partially mitigate the failure. `MultiSpeciesDiffusionSolver.set_initial_state` and `PUSLFDAdvectionDiffusionSolver.bdf3_step` remain clear hybrid failures despite strong dense ranks.

# Conclusions

| Status | Targets |
|--------|---------|
| BM25 failure (persistent) | PUSLAdvectionSolver.forward_sl_step, PUSLFDAdvectionDiffusionSolver.bdf3_step |
| BM25 failure (resolved by summaries) | DiffusionSolver.set_state_history |
| Reranker problems | VariablePoissonSolver.solve, NonlinearVariablePoissonSolver.solve, IncompressibleEulerBDFBackend._get_cached_system |

Note: "resolved by summaries" refers to recall@3 eval hits, not wide hybrid rank.

**Fix for BM25:** tokenize on camelCase and underscore boundaries — `forward_sl_step` → `["forward", "sl", "step"]`. This is the change the primary notebook evaluates.


In [67]:
print(f"── v1 recall@3, 5, 10 ──────────")
for k in [3, 5, 10]:
    hits, totals = run_eval(qa_pairs_solvers, idx_v1, k=k)
    total_hits = sum(hits.values())
    total_pairs = sum(totals.values())
    print(f"k = {k}: {total_hits}/{total_pairs}")

print(f"── v2 recall@3, 5, 10 ──────────")
for k in [3, 5, 10]:
    hits, totals = run_eval(qa_pairs_solvers, idx_v2, k=k)
    total_hits = sum(hits.values())
    total_pairs = sum(totals.values())
    print(f"k = {k}: {total_hits}/{total_pairs}")

print(f"── v3a recall@3, 5, 10 ──────────")
for k in [3, 5, 10]:
    hits, totals = run_eval(qa_pairs_solvers, idx_v3a, k=k)
    total_hits = sum(hits.values())
    total_pairs = sum(totals.values())
    print(f"k = {k}: {total_hits}/{total_pairs}")

print(f"── v3 recall@3, 5, 10 ──────────")
for k in [3, 5, 10]:
    hits, totals = run_eval(qa_pairs_solvers, idx_v3, k=k)
    total_hits = sum(hits.values())
    total_pairs = sum(totals.values())
    print(f"k = {k}: {total_hits}/{total_pairs}")

── v1 recall@3, 5, 10 ──────────
k = 3: 1/10
k = 5: 4/10
k = 10: 5/10
── v2 recall@3, 5, 10 ──────────
k = 3: 1/10
k = 5: 1/10
k = 10: 2/10
── v3a recall@3, 5, 10 ──────────
k = 3: 2/10
k = 5: 4/10
k = 10: 5/10
── v3 recall@3, 5, 10 ──────────
k = 3: 4/10
k = 5: 5/10
k = 10: 6/10
